# سبق 10 - پیداوار میں AI ایجنٹس

اس سبق میں آپ Microsoft Agent Framework (Python) استعمال کرتے ہوئے AI ایجنٹس کے لیے **پیداواری نمونے** سیکھیں گے۔ ہم ان موضوعات کا احاطہ کرتے ہیں:

- **قابلِ مشاہدہ** — ایجنٹ کے تعاملات میں ٹائمنگ اور لاگنگ شامل کرنا
- **تشخیص** — ردِعمل کے معیار کو اسکور کرنے کے لیے ایک تشخیصی ایجنٹ کا استعمال
- **لاگت کا انتظام** — ٹوکن کی بہتر کاری اور ماڈل کے انتخاب کی حکمتِ عملیاں

منظر نامہ ایک **سفر ایجنٹ** ہے جو صارفین کو سفر کی منصوبہ بندی میں مدد دیتا ہے، جس پر مانیٹرنگ اور تشخیص کی تہہ رکھی گئی ہے۔


## ترتیب


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## پیداوار کے لیے غور و فکر

AI ایجنٹس کو پروٹوٹائپس سے پیداوار میں منتقل کرنے کے لیے تین ستونوں پر محتاط توجہ درکار ہے:

1. **نظارت** — آپ کو اس بات کی مرئیت چاہیے کہ ایجنٹ کیا کر رہا ہے، اسے کتنا وقت لگ رہا ہے، اور وہ کون سے ٹولز کو کال کر رہا ہے۔ ٹریسنگ اور لاگنگ کے بغیر، پیداوار میں مسائل کو ڈیبگ کرنا تقریباً ناممکن ہے۔

2. **تشخیص** — خودکار معیار چیک اس بات کو یقینی بناتے ہیں کہ وقت کے ساتھ ایجنٹ کے جوابات درست، مکمل، اور مددگار رہیں۔ ایک تشخیصی ایجنٹ مخصوص معیار کے خلاف جوابات کو اسکور کر سکتا ہے۔

3. **لاگت کا انتظام** — ٹوکن کے استعمال کا براہِ راست اثر لاگت پر پڑتا ہے۔ پرومپٹ کی بہتر کاری، ماڈل کے انتخاب، اور کیشنگ جیسی حکمتِ عملیاں بغیر معیار قربان کیے اخراجات کو قابو میں رکھنے میں مدد دیتی ہیں۔


## ایک قابلِ مشاہدہ ایجنٹ بنانا

ہم سفر کے اوزار متعین کرتے ہیں اور ایجنٹ کال کو ٹائمنگ کے ساتھ لپیٹتے ہیں تاکہ ہم تاخیر کی نگرانی کر سکیں۔ پروڈکشن میں آپ OpenTelemetry یا اسی طرح کے ٹریسنگ بیک اینڈ کے ساتھ مربوط کریں گے۔


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## تشخیص کے نمونے

ایک عام پیداوار کا طریقہ یہ ہے کہ دوسرے ایجنٹ کو بطور **جانچنے والا** استعمال کیا جائے۔ جانچنے والا بنیادی ایجنٹ کے جواب کو پہلے سے متعین معیار جیسے مکملیت، درستگی، اور مفیدیت کے مطابق اسکور کرتا ہے۔

اس سے مندرجہ ذیل ممکن ہوتا ہے:
- جوابات صارفین تک پہنچنے سے پہلے خودکار معیار کنٹرول
- جب پرامپٹس یا ماڈلز تبدیل ہوں تو ریگریشن کا پتہ لگانا
- وقت کے ساتھ ایجنٹ کی کارکردگی کی مسلسل نگرانی


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## لاگت کے انتظام کی حکمت عملیاں

لاگتوں کو کنٹرول کرنا پروڈکشن AI ایجنٹس کے لیے انتہائی ضروری ہے۔ یہاں کلیدی حکمت عملیاں درج ہیں:

| حکمتِ عملی | تفصیل |
|---|---|
| **پرومپٹ کی اصلاح** | سسٹم ہدایات کو مختصر رکھیں۔ غیر ضروری سیاق و سباق کو ہٹا کر ان پٹ ٹوکنز کم کریں۔ |
| **ماڈل کا انتخاب** | سادہ کاموں جیسے درجہ بندی یا استخراج کے لیے چھوٹے، سستے ماڈلز (مثلاً GPT-4o-mini) استعمال کریں، اور پیچیدہ استدلال کے لیے بڑے ماڈلز کو مخصوص رکھیں۔ |
| **کیشنگ** | ٹول کے نتائج اور بار بار کیے جانے والے سوالات کو کیش کریں تاکہ غیر ضروری API کالز سے بچا جا سکے۔ |
| **ٹوکن بجٹس** | غیر متوقع طور پر طویل جوابات کو روکنے کے لیے `max_tokens` حدود مقرر کریں۔ |
| **بیچنگ** | جہاں ممکن ہو متعدد صارف کے سوالات کو ایک واحد API کال میں گروپ کریں۔ |

عملی طور پر، ایک مرحلہ وار طریقہ کار مؤثر ثابت ہوتا ہے: سیدھے سادے درخواستوں کو ایک تیز، سستا ماڈل بھیجیں اور صرف پیچیدہ سوالات کو ہی ایک زیادہ قابل ماڈل کی طرف بڑھائیں۔


## خلاصہ

اس سبق میں آپ نے سیکھا کہ آپ کیسے:

1. **مشاہدہ شامل کریں** ایجنٹ تعاملات میں وقت بندی اور لاگنگ کے ذریعے، ٹریسنگ اور مانیٹرنگ کے لیے بنیاد قائم کرتے ہوئے۔
2. **ایجنٹ کے جوابات کا جائزہ لیں** خودکار طور پر ایک ایویلیوایٹر ایجنٹ استعمال کرتے ہوئے جو مکملیت، درستگی، اور مفیدیت کو اسکور کرتا ہے۔
3. **اخراجات کا انتظام کریں** پرامپٹ کی اصلاح، ماڈل کے انتخاب، کیشنگ، اور ٹوکن بجٹس کے ذریعے۔

یہ پروڈکشن پیٹرنز اس بات کو یقینی بنانے میں مدد دیتے ہیں کہ آپ کے مصنوعی ذہانت کے ایجنٹس بڑے پیمانے پر قابلِ اعتماد، قابلِ پیمائش، اور لاگت کے لحاظ سے موثر ہوں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ڈسکلیمر:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) استعمال کرتے ہوئے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہِ کرم نوٹ کریں کہ خودکار تراجم میں غلطیاں یا عدمِ درستی ہوسکتی ہے۔ اصل دستاویز اپنی مادری زبان میں ہی معتبر ماخذ مانی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ورانہ انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کے لیے ہم ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
